In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:

!git clone https://github.com/kashish-0409/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System.git


Cloning into 'NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System'...
remote: Enumerating objects: 14335, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 14335 (delta 35), reused 18 (delta 17), pack-reused 14286 (from 1)
Receiving objects: 100% (14335/14335), 391.50 MiB | 31.86 MiB/s, done.
Resolving deltas: 100% (91/91), done.
Updating files: 100% (18759/18759), done.


In [4]:

import os
from pathlib import Path

ROOT = Path(
    "/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System"
)

print("Repository exists:", ROOT.exists())

for folder in [
    "Processed Data",
    "NourishIQ_Dataset_Split",
    "NourishIQ_Split_Dataset",
    "models",
    "results"
]:
    path = ROOT / folder
    print(folder, ":", path.exists())

images = [
    p for p in (ROOT / "Processed Data").rglob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]
]

print("\nTotal downloaded images:", len(images))
print("Sample image:", images[0] if images else "NONE")

print("\nCSV files:")
for p in ROOT.rglob("*.csv"):
    print(p)


Repository exists: True
Processed Data : True
NourishIQ_Dataset_Split : True
NourishIQ_Split_Dataset : True
models : True
results : True

Total downloaded images: 14160
Sample image: /kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/Processed Data/Rotten Cucumber(12-20)/aug_193_IMG_20251202_015425.jpg

CSV files:
/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/dataset_split.csv
/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/NourishIQ_Dataset_Split/dataset_split.csv
/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/results/clean_dataset_split.csv
/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/results/mobilenetv2_confusion_matrix.csv
/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/results/mobilenetv2_history.csv
/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-Sys

In [5]:
!pip install -q timm==1.0.15

In [6]:

import os
import re
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# =====================================================
# 1. CONFIGURATION
# =====================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

ROOT = Path(
    "/kaggle/working/"
    "NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System"
)

DATA_DIR = ROOT / "Processed Data"
CSV_PATH = ROOT / "results" / "clean_dataset_split.csv"

OUTPUT_DIR = ROOT / "results" / "convnextv2_base"
MODEL_DIR = ROOT / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2

EPOCHS_HEAD = 8
EPOCHS_FINE = 20
PATIENCE = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.device_count() < 1:
    raise RuntimeError("GPU not available. Enable GPU in Kaggle.")

# =====================================================
# 2. LOAD CLEAN DATASET SPLIT
# =====================================================

df = pd.read_csv(CSV_PATH)

print("\nCSV columns:", df.columns.tolist())

def find_column(possible_names):
    for col in df.columns:
        if col.lower().strip() in possible_names:
            return col
    return None

class_col = find_column({
    "class", "class_name", "label", "category"
})

filename_col = find_column({
    "filename", "file_name", "image",
    "image_name", "image_path", "filepath", "path"
})

split_col = find_column({
    "split", "set", "dataset_split", "subset"
})

if not all([class_col, filename_col, split_col]):
    raise ValueError(
        "Could not detect required CSV columns. "
        f"Detected: class={class_col}, "
        f"filename={filename_col}, split={split_col}. "
        f"Available columns: {df.columns.tolist()}"
    )

df = df[[class_col, filename_col, split_col]].copy()
df.columns = ["class_name", "filename", "split"]

df["class_name"] = df["class_name"].astype(str).str.strip()
df["split"] = df["split"].astype(str).str.lower().str.strip()

split_mapping = {
    "train": "train",
    "training": "train",
    "val": "validation",
    "valid": "validation",
    "validation": "validation",
    "test": "test",
    "testing": "test"
}

df["split"] = df["split"].map(split_mapping)

df = df.dropna(subset=["class_name", "filename", "split"])

classes = sorted(df["class_name"].unique())
class_to_idx = {name: i for i, name in enumerate(classes)}

print("\nNumber of classes:", len(classes))
print("Classes:", classes)

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

print("\nTrain:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

if len(classes) != 24:
    raise ValueError(
        f"Expected 24 classes but found {len(classes)}. "
        "Check the CSV before training."
    )

if min(len(train_df), len(val_df), len(test_df)) == 0:
    raise ValueError("One of the dataset splits is empty.")

# =====================================================
# 3. RESOLVE IMAGE PATHS
# =====================================================

def resolve_image_path(row):
    raw = str(row["filename"]).replace("\\", "/")
    cls = row["class_name"]

    candidates = [
        Path(raw),
        ROOT / raw,
        DATA_DIR / cls / Path(raw).name,
        DATA_DIR / raw
    ]

    for path in candidates:
        if path.exists() and path.is_file():
            return str(path)

    return None


for part_name, part_df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    part_df["image_path"] = part_df.apply(
        resolve_image_path, axis=1
    )

    missing = part_df["image_path"].isna().sum()
    print(part_name, "missing images:", missing)

    if missing:
        raise FileNotFoundError(
            f"{missing} missing images in {part_name} split."
        )

# =====================================================
# 4. IMAGE TRANSFORMS
# =====================================================

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(12),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# =====================================================
# 5. PYTORCH DATASET
# =====================================================

class NourishDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        image = self.transform(image)

        label = class_to_idx[row["class_name"]]

        return image, label


train_dataset = NourishDataset(train_df, train_transform)
val_dataset = NourishDataset(val_df, eval_transform)
test_dataset = NourishDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# =====================================================
# 6. LOAD PRETRAINED CONVNEXTV2-BASE
# =====================================================

model = timm.create_model(
    "convnextv2_base.fcmae_ft_in22k_in1k",
    pretrained=True,
    num_classes=len(classes)
)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

model = model.to(DEVICE)

print("\nConvNeXtV2-Base loaded successfully.")

# =====================================================
# 7. TRAINING UTILITIES
# =====================================================

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

def get_base_model():
    return model.module if isinstance(
        model, nn.DataParallel
    ) else model


def run_epoch(loader, optimizer=None, scaler=None):
    training = optimizer is not None

    model.train(training)

    total_loss = 0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(all_labels, all_preds)

    return avg_loss, accuracy


# =====================================================
# 8. STAGE 1 - TRAIN CLASSIFICATION HEAD
# =====================================================

base_model = get_base_model()

for parameter in base_model.parameters():
    parameter.requires_grad = False

for parameter in base_model.head.parameters():
    parameter.requires_grad = True

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-4,
    weight_decay=1e-4
)

scaler = torch.cuda.amp.GradScaler()

best_val = 0.0
best_path = OUTPUT_DIR / "best_convnextv2.pth"

print("\n========== STAGE 1: FROZEN BACKBONE ==========")

for epoch in range(EPOCHS_HEAD):

    train_loss, train_acc = run_epoch(
        train_loader, optimizer, scaler
    )

    val_loss, val_acc = run_epoch(val_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS_HEAD} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_acc > best_val:
        best_val = val_acc

        torch.save({
            "model_state": get_base_model().state_dict(),
            "classes": classes,
            "best_val": best_val
        }, best_path)

# =====================================================
# 9. STAGE 2 - FINE TUNING
# =====================================================

checkpoint = torch.load(
    best_path, map_location=DEVICE, weights_only=False
)

get_base_model().load_state_dict(checkpoint["model_state"])

for parameter in get_base_model().parameters():
    parameter.requires_grad = True

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.3,
    patience=2,
    min_lr=1e-7
)

best_val = checkpoint["best_val"]
wait = 0

print("\n========== STAGE 2: FINE TUNING ==========")

for epoch in range(EPOCHS_FINE):

    train_loss, train_acc = run_epoch(
        train_loader, optimizer, scaler
    )

    val_loss, val_acc = run_epoch(val_loader)

    scheduler.step(val_acc)

    print(
        f"Epoch {epoch+1}/{EPOCHS_FINE} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_acc > best_val:
        best_val = val_acc
        wait = 0

        torch.save({
            "model_state": get_base_model().state_dict(),
            "classes": classes,
            "best_val": best_val
        }, best_path)

        print("Best checkpoint saved.")

    else:
        wait += 1

    if wait >= PATIENCE:
        print("Early stopping.")
        break

# =====================================================
# 10. LOAD BEST CHECKPOINT
# =====================================================

checkpoint = torch.load(
    best_path, map_location=DEVICE, weights_only=False
)

get_base_model().load_state_dict(checkpoint["model_state"])

print("\nBest validation accuracy:", checkpoint["best_val"])

# =====================================================
# 11. FINAL TEST EVALUATION
# =====================================================

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.cuda.amp.autocast():
            outputs = model(images)

        predictions = outputs.argmax(dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = accuracy_score(all_labels, all_preds)

macro_f1 = f1_score(
    all_labels, all_preds, average="macro"
)

weighted_f1 = f1_score(
    all_labels, all_preds, average="weighted"
)

report = classification_report(
    all_labels,
    all_preds,
    target_names=classes,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

report_df.to_csv(
    OUTPUT_DIR / "classification_report.csv"
)

cm = confusion_matrix(all_labels, all_preds)

pd.DataFrame(
    cm,
    index=classes,
    columns=classes
).to_csv(OUTPUT_DIR / "confusion_matrix.csv")

# Save final selected model
final_path = MODEL_DIR / "nourishiq_convnextv2_base_best.pth"

torch.save({
    "model_state": get_base_model().state_dict(),
    "classes": classes,
    "image_size": IMAGE_SIZE,
    "test_accuracy": test_accuracy,
    "best_val_accuracy": best_val
}, final_path)

with open(
    OUTPUT_DIR / "class_labels.json", "w"
) as f:
    json.dump(classes, f, indent=4)

print("\n======================================")
print("CONVNEXTV2-BASE TRAINING COMPLETED")
print("Best validation accuracy:", round(best_val * 100, 2), "%")
print("TEST ACCURACY:", round(test_accuracy * 100, 2), "%")
print("Macro F1:", round(macro_f1, 4))
print("Weighted F1:", round(weighted_f1, 4))
print("Saved model:", final_path)
print("Reports:", OUTPUT_DIR)
print("======================================")

print(report_df)

Device: cuda
GPU count: 2

CSV columns: ['image_path', 'class', 'split']

Number of classes: 24
Classes: ['Fresh Banana(1-4)', 'Fresh Bittermelon(1-3)', 'Fresh Cucumber(1-6)', 'Fresh Orange(1-9)', 'Fresh Papaya(1-4)', 'Fresh Tomato(1-10)', 'Fresh eggplant(1-4)', 'Fresh pineapple(1-15)', 'Rotten Bittermelon(5-8)', 'Rotten Cucumber(12-20)', 'Rotten Orange(20-35)', 'Rotten Papaya(7-12)', 'Rotten Pineapple(25-35)', 'Rotten Tomato(24-35)', 'Rotten banana(7-13)', 'Rotten eggplant(8-15)', 'Semi Fresh Bittermelon ( 3-5)', 'Semi Fresh Cucumber(6-12)', 'Semi Fresh Papaya(4-7)', 'Semi fresh Orange(9-20)', 'Semi fresh Pineapple (15-25)', 'Semi fresh Tomato(10-24)', 'Semi fresh banana(4-7)', 'Semi_Fresh eggplant(4-8)']

Train: 9829
Validation: 1997
Test: 2211
train missing images: 0
validation missing images: 0
test missing images: 0


model.safetensors: reconstructing file:   0%|          |  0.00B /  355MB            

model.safetensors: downloading bytes:           |  0.00B            


ConvNeXtV2-Base loaded successfully.

========== STAGE 1: FROZEN BACKBONE ==========


/tmp/ipykernel_47/3136717604.py:349: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_47/3136717604.py:309: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_47/3136717604.py:309: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/8 | Train Acc: 0.7970 | Val Acc: 0.8798 | Val Loss: 0.7035


... (50 outputs left out)

/tmp/ipykernel_47/3136717604.py:309: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_47/3136717604.py:309: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 19/20 | Train Acc: 1.0000 | Val Acc: 0.9579 | Val Loss: 0.5062
Early stopping.

Best validation accuracy: 0.9589384076114171


/tmp/ipykernel_47/3136717604.py:476: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



CONVNEXTV2-BASE TRAINING COMPLETED
Best validation accuracy: 95.89 %
TEST ACCURACY: 96.07 %
Macro F1: 0.9625
Weighted F1: 0.9593
Saved model: /kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/models/nourishiq_convnextv2_base_best.pth
Reports: /kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/results/convnextv2_base
                               precision    recall  f1-score      support
Fresh Banana(1-4)               1.000000  0.913043  0.954545    92.000000
Fresh Bittermelon(1-3)          0.989583  1.000000  0.994764    95.000000
Fresh Cucumber(1-6)             1.000000  1.000000  1.000000   109.000000
Fresh Orange(1-9)               1.000000  1.000000  1.000000    89.000000
Fresh Papaya(1-4)               0.776786  0.956044  0.857143    91.000000
Fresh Tomato(1-10)              0.988764  1.000000  0.994350    88.000000
Fresh eggplant(1-4)             0.970000  0.960396  0.965174   101.000000
Fresh pineapple(1-15)     

In [7]:
from pathlib import Path

ROOT = Path(
    "/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System"
)

checkpoint = ROOT / "results/convnextv2_base/best_convnextv2.pth"

print("Checkpoint exists:", checkpoint.exists())

if checkpoint.exists():
    print("Checkpoint size (MB):", round(checkpoint.stat().st_size / (1024**2), 2))

Checkpoint exists: True
Checkpoint size (MB): 334.76


In [8]:

import torch
import torch.nn as nn
import timm
import json
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score
)

ROOT = Path(
    "/kaggle/working/"
    "NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System"
)

CHECKPOINT = ROOT / "results/convnextv2_base/best_convnextv2.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load checkpoint
ckpt = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

classes = ckpt["classes"]

print("Best validation accuracy:", ckpt["best_val"] * 100)
print("Number of classes:", len(classes))

# Rebuild same architecture
model = timm.create_model(
    "convnextv2_base.fcmae_ft_in22k_in1k",
    pretrained=False,
    num_classes=len(classes)
)

model.load_state_dict(ckpt["model_state"])
model = model.to(device)
model.eval()

# Use the existing test_loader from your earlier code
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.cuda.amp.autocast():
            outputs = model(images)

        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)
macro_f1 = f1_score(
    all_labels, all_preds, average="macro"
)
weighted_f1 = f1_score(
    all_labels, all_preds, average="weighted"
)

print("\n========== FINAL TEST RESULTS ==========")
print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

print("\nCLASSIFICATION REPORT:")
print(classification_report(
    all_labels,
    all_preds,
    target_names=classes,
    zero_division=0
))


Best validation accuracy: 95.89384076114172
Number of classes: 24


/tmp/ipykernel_47/902800984.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



========== FINAL TEST RESULTS ==========
Test Accuracy: 96.07%
Macro F1: 0.9625
Weighted F1: 0.9593

CLASSIFICATION REPORT:
                               precision    recall  f1-score   support

            Fresh Banana(1-4)       1.00      0.91      0.95        92
       Fresh Bittermelon(1-3)       0.99      1.00      0.99        95
          Fresh Cucumber(1-6)       1.00      1.00      1.00       109
            Fresh Orange(1-9)       1.00      1.00      1.00        89
            Fresh Papaya(1-4)       0.78      0.96      0.86        91
           Fresh Tomato(1-10)       0.99      1.00      0.99        88
          Fresh eggplant(1-4)       0.97      0.96      0.97       101
        Fresh pineapple(1-15)       0.92      1.00      0.96       112
      Rotten Bittermelon(5-8)       1.00      1.00      1.00        95
       Rotten Cucumber(12-20)       1.00      1.00      1.00        99
         Rotten Orange(20-35)       1.00      1.00      1.00       100
          Rotten Papay

In [9]:

import pandas as pd
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ROOT = Path(
    "/kaggle/working/"
    "NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System"
)

DATA_DIR = ROOT / "Processed Data"
CSV_PATH = ROOT / "results" / "clean_dataset_split.csv"

df = pd.read_csv(CSV_PATH)

print("CSV columns:", df.columns.tolist())

# Detect columns
def find_col(names):
    for col in df.columns:
        if col.lower().strip() in names:
            return col
    raise ValueError(f"Column not found: {names}")

class_col = find_col({"class", "class_name", "label", "category"})
file_col = find_col({
    "filename", "file_name", "image",
    "image_name", "image_path", "filepath", "path"
})
split_col = find_col({"split", "set", "dataset_split", "subset"})

test_df = df[[class_col, file_col, split_col]].copy()
test_df.columns = ["class_name", "filename", "split"]

test_df["split"] = test_df["split"].astype(str).str.lower().str.strip()

test_df = test_df[
    test_df["split"].isin(["test", "testing"])
].copy()

# Use class order saved in checkpoint
class_to_idx = {name: i for i, name in enumerate(classes)}

def resolve_path(row):
    raw = str(row["filename"]).replace("\\", "/")
    cls = row["class_name"]

    candidates = [
        Path(raw),
        ROOT / raw,
        DATA_DIR / cls / Path(raw).name,
        DATA_DIR / raw
    ]

    for p in candidates:
        if p.exists() and p.is_file():
            return str(p)

    return None

test_df["image_path"] = test_df.apply(resolve_path, axis=1)

missing = test_df["image_path"].isna().sum()

print("Test images:", len(test_df))
print("Missing images:", missing)

if missing:
    raise FileNotFoundError(
        f"{missing} test images could not be located."
    )

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class TestDataset(Dataset):
    def __init__(self, data):
        self.data = data.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        image = transform(image)
        label = class_to_idx[row["class_name"]]
        return image, label

test_dataset = TestDataset(test_df)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Test DataLoader ready:", len(test_dataset))


CSV columns: ['image_path', 'class', 'split']
Test images: 2211
Missing images: 0
Test DataLoader ready: 2211


In [10]:
from pathlib import Path

ROOT = Path("/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System")

for p in ROOT.rglob("*"):
    if p.is_file() and (
        "convnext" in p.name.lower()
        or "history" in p.name.lower()
        or "log" in p.name.lower()
    ):
        print(p)

/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/results/mobilenetv2_history.csv
/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/results/mobilenetv2_finetuned_history.csv
/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/results/training_history.csv
/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/results/clean_mobilenetv2_history.csv
/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/models/nourishiq_convnextv2_base_best.pth
/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/results/convnextv2_base/best_convnextv2.pth


In [11]:

from pathlib import Path
import pandas as pd

ROOT = Path(
    "/kaggle/working/"
    "NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System"
)

files = [
    ROOT / "results/training_history.csv",
    ROOT / "results/efficientnetv2/training_history.csv",
    ROOT / "results/efficientnetv2b2/training_history.csv",
    ROOT / "results/mobilenetv2_history.csv",
    ROOT / "results/clean_mobilenetv2_history.csv",
    ROOT / "results/mobilenetv2_finetuned_history.csv"
]

for file in files:
    if file.exists():
        df = pd.read_csv(file)
        print("\nFILE:", file.name)
        print("Columns:", df.columns.tolist())
        print("Rows:", len(df))
        print(df.head(3).to_string(index=False))



FILE: training_history.csv
Columns: ['epoch', 'accuracy', 'loss', 'val_accuracy', 'val_loss']
Rows: 15
 epoch  accuracy     loss  val_accuracy  val_loss
     0  0.136098 2.728017      0.356061  2.012995
     1  0.331114 1.931627      0.433239  1.636902
     2  0.413136 1.681267      0.523201  1.420885

FILE: mobilenetv2_history.csv
Columns: ['epoch', 'accuracy', 'loss', 'val_accuracy', 'val_loss']
Rows: 10
 epoch  accuracy     loss  val_accuracy  val_loss
     0  0.777139 0.649787      0.875947  0.312572
     1  0.902341 0.273021      0.921402  0.221591
     2  0.927159 0.204800      0.931345  0.190818

FILE: clean_mobilenetv2_history.csv
Columns: ['epoch', 'accuracy', 'loss', 'val_accuracy', 'val_loss']
Rows: 3
 epoch  accuracy     loss  val_accuracy  val_loss
     0  0.955845 0.121775      0.881322  0.407617
     1  0.966426 0.096137      0.890336  0.381727
     2  0.971615 0.079714      0.885829  0.445073

FILE: mobilenetv2_finetuned_history.csv
Columns: ['epoch', 'accuracy', 'loss

In [12]:
#python -m pip install streamlit torch torchvision timm numpy pandas Pillow

In [13]:
!pip install -q streamlit timm

In [14]:
import torch
import torchvision
import timm
import streamlit
import numpy
import pandas
from PIL import Image

print("All required libraries imported successfully!")
print("PyTorch:", torch.__version__)
print("Timm:", timm.__version__)

All required libraries imported successfully!
PyTorch: 2.11.0+cu130
Timm: 1.0.15


In [26]:

app_code = r'''
import os
import io
import re
import streamlit as st
import torch
import timm
import numpy as np

from PIL import Image, ImageOps
from torchvision import transforms

# =====================================================
# CONFIGURATION
# =====================================================

st.set_page_config(
    page_title="NourishIQ | Food Freshness Intelligence",
    page_icon="🍃",
    layout="wide",
    initial_sidebar_state="collapsed"
)

MODEL_PATH = (
    "/kaggle/working/"
    "NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/"
    "results/convnextv2_base/best_convnextv2.pth"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =====================================================
# PREMIUM UI DESIGN
# =====================================================

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&family=Manrope:wght@500;600;700;800&display=swap');

.stApp {
    background: #F5F5ED;
    color: #183C2D;
    font-family: 'DM Sans', sans-serif;
}

.block-container {
    max-width: 1120px;
    padding: 1.8rem 2rem 3rem;
}

#MainMenu, footer {
    visibility: hidden;
}

header[data-testid="stHeader"] {
    background: transparent;
}

.brand {
    font-family: 'Manrope', sans-serif;
    font-size: 2.1rem;
    font-weight: 800;
    color: #17452F;
    letter-spacing: -1.3px;
}

.brand span {
    color: #91AC62;
}

.tagline {
    font-size: .68rem;
    font-weight: 700;
    letter-spacing: 3px;
    color: #82917D;
    margin-top: -5px;
}

.hero {
    background: linear-gradient(120deg, #153D2D, #285D43);
    padding: 43px 42px;
    border-radius: 25px;
    margin: 24px 0 18px;
    box-shadow: 0 12px 30px rgba(23,65,45,.12);
}

.hero h1 {
    color: #FFFFFF;
    font-family: 'Manrope', sans-serif;
    font-size: clamp(2rem, 4vw, 3.1rem);
    font-weight: 800;
    line-height: 1.12;
    margin: 0 0 14px;
}

.hero p {
    color: #DCE9DB;
    font-size: 1rem;
    max-width: 650px;
    margin: 0;
}

.eyebrow {
    color: #AFC995;
    font-size: .72rem;
    letter-spacing: 2.5px;
    font-weight: 800;
    margin-bottom: 13px;
}

.section-title {
    color: #193D2D;
    font-family: 'Manrope', sans-serif;
    font-size: 1.45rem;
    font-weight: 800;
    margin: 25px 0 4px;
}

.muted {
    color: #788678;
    font-size: .9rem;
}

.upload-card {
    background: #FFFFFF;
    border: 1px solid #DCE6D7;
    border-radius: 19px;
    padding: 20px;
    margin-top: 14px;
}

.result-card {
    background: #FFFFFF;
    border: 1px solid #DCE6D7;
    border-radius: 18px;
    padding: 19px 17px;
    min-height: 112px;
    box-shadow: 0 5px 18px rgba(30,65,44,.04);
}

.result-label {
    color: #81907D;
    font-size: .76rem;
    font-weight: 600;
    margin-bottom: 8px;
}

.result-value {
    color: #173F2D;
    font-family: 'Manrope', sans-serif;
    font-size: 1.25rem;
    font-weight: 800;
    overflow-wrap: anywhere;
}

.fresh-status {
    color: #247444;
}

.semi-status {
    color: #B17A27;
}

.rotten-status {
    color: #B34E45;
}

.prediction-row {
    background: #FFFFFF;
    border: 1px solid #E1E8DD;
    border-radius: 13px;
    padding: 13px 15px;
    margin: 10px 0;
}

div[data-testid="stFileUploader"] {
    background: #FFFFFF;
    border: 1px dashed #94AE8D;
    border-radius: 17px;
    padding: 14px;
}

div.stButton > button {
    background: #23583D;
    color: white;
    border: none;
    border-radius: 12px;
    min-height: 49px;
    font-weight: 700;
}

div.stButton > button:hover {
    background: #17432E;
    color: white;
    border: none;
}

div[data-testid="stMetric"] {
    background: white;
    border: 1px solid #E1E8DD;
    border-radius: 15px;
    padding: 15px;
}

div[data-testid="stExpander"] {
    background: #FFFFFF;
    border: 1px solid #DCE6D7;
    border-radius: 14px;
}

div[data-testid="stImage"] img {
    border-radius: 16px;
}

.footer {
    text-align: center;
    color: #829080;
    font-size: .8rem;
    padding: 24px 0 8px;
}

hr {
    border-color: #DDE5D8;
}

</style>
""", unsafe_allow_html=True)


# =====================================================
# HEADER
# =====================================================

st.markdown("""
<div class="brand">🍃 Nourish<span>IQ</span></div>
<div class="tagline">INTELLIGENT FOOD FRESHNESS</div>

<div class="hero">
    <div class="eyebrow">AI-POWERED FOOD INTELLIGENCE</div>
    <h1>Know your food.<br>Make informed choices.</h1>
    <p>
        Discover visible freshness stages through intelligent
        computer vision. Upload a fruit image and let AI analyse it.
    </p>
</div>
""", unsafe_allow_html=True)

st.markdown(
    '<p class="muted">A computer-vision prototype for preliminary '
    'food freshness assessment. Predictions cannot confirm food '
    'safety or exact shelf life.</p>',
    unsafe_allow_html=True
)


# =====================================================
# MODEL LOADING
# =====================================================

@st.cache_resource
def load_model():

    if not os.path.isfile(MODEL_PATH):
        raise FileNotFoundError(
            f"Checkpoint not found: {MODEL_PATH}"
        )

    checkpoint = torch.load(
        MODEL_PATH,
        map_location=DEVICE,
        weights_only=False
    )

    classes = checkpoint["classes"]

    model = timm.create_model(
        "convnextv2_base.fcmae_ft_in22k_in1k",
        pretrained=False,
        num_classes=len(classes)
    )

    model.load_state_dict(checkpoint["model_state"])
    model.to(DEVICE)
    model.eval()

    return model, classes


try:
    model, class_names = load_model()

except Exception as error:
    st.error(f"Model loading failed: {error}")
    st.stop()


# =====================================================
# IMAGE PREPROCESSING
# =====================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# =====================================================
# LABEL PROCESSING
# =====================================================

def clean_label(label):

    # Remove metadata such as (1-6), (25-35)
    label = re.sub(r"\s*\([^)]*\)", "", str(label))

    return label.replace("_", " ").strip()


def parse_prediction(label):

    normalized = clean_label(label)
    parts = normalized.split()

    if (
        len(parts) >= 2
        and parts[0].lower() == "semi"
        and parts[1].lower() == "fresh"
    ):
        return " ".join(parts[2:]).strip(), "Semi Fresh"

    if parts and parts[0].lower() in ("fresh", "rotten"):
        return " ".join(parts[1:]).strip(), parts[0].title()

    return normalized, "Unclassified"


# =====================================================
# MODEL PERFORMANCE
# =====================================================

with st.expander("📊 View Model Performance"):

    st.markdown("### ConvNeXtV2-Base Evaluation")

    m1, m2, m3, m4 = st.columns(4)

    m1.metric("Test Accuracy", "96.07%")
    m2.metric("Macro F1-score", "96.25%")
    m3.metric("Weighted F1-score", "95.93%")
    m4.metric("Test Images", "2,211")

    st.caption(
        "These metrics represent evaluation on the saved test dataset. "
        "They are not the confidence score of an individual prediction."
    )


# =====================================================
# IMAGE UPLOAD
# =====================================================

st.markdown(
    '<div class="section-title">01 · Analyse your food</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<p class="muted">Upload a clear image containing one fruit '
    'for freshness classification.</p>',
    unsafe_allow_html=True
)

uploaded_file = st.file_uploader(
    "Choose your fruit image",
    type=["jpg", "jpeg", "png", "webp"],
    help="Supported formats: JPG, JPEG, PNG and WEBP."
)


# =====================================================
# IMAGE PREVIEW AND PREDICTION
# =====================================================

if uploaded_file is not None:

    try:
        image = Image.open(
            io.BytesIO(uploaded_file.getvalue())
        )

        image = ImageOps.exif_transpose(image).convert("RGB")

        st.markdown(
            '<div class="section-title">02 · Image workspace</div>',
            unsafe_allow_html=True
        )

        left, right = st.columns([1, 1.1], gap="large")

        with left:
            st.image(
                image,
                caption="Uploaded fruit image",
                use_container_width=True
            )

        with right:
            st.markdown("### Ready to analyse?")
            st.markdown(
                '<p class="muted">ConvNeXtV2-Base will analyse '
                'the image and estimate its visible freshness stage.</p>',
                unsafe_allow_html=True
            )

            if st.button(
                "✦  Analyse Freshness",
                type="primary",
                use_container_width=True
            ):

                with st.spinner("Analysing your image..."):

                    tensor = transform(image).unsqueeze(0).to(DEVICE)

                    with torch.inference_mode():
                        logits = model(tensor)
                        probabilities = torch.softmax(logits, dim=1)[0]

                    values, indices = torch.topk(
                        probabilities,
                        k=min(5, len(class_names))
                    )

                    predicted_index = int(indices[0].item())
                    predicted_label = class_names[predicted_index]

                    food, freshness = parse_prediction(
                        predicted_label
                    )

                    confidence = float(values[0].item())

                    st.session_state["nourish_result"] = {
                        "food": food,
                        "freshness": freshness,
                        "confidence": confidence,
                        "indices": indices.cpu().tolist(),
                        "values": values.cpu().tolist()
                    }

    except Exception as error:
        st.error(f"Image processing failed: {error}")


# =====================================================
# RESULTS
# =====================================================

result = st.session_state.get("nourish_result")

if result:

    freshness = result["freshness"]

    status_class = {
        "Fresh": "fresh-status",
        "Semi Fresh": "semi-status",
        "Rotten": "rotten-status"
    }.get(freshness, "")

    st.markdown("---")

    st.markdown(
        '<div class="section-title">03 · Your freshness analysis</div>',
        unsafe_allow_html=True
    )

    a, b, c = st.columns(3)

    with a:
        st.markdown(
            f'<div class="result-card">'
            f'<div class="result-label">PREDICTED FRUIT</div>'
            f'<div class="result-value">{result["food"] or "Unknown"}</div>'
            f'</div>',
            unsafe_allow_html=True
        )

    with b:
        st.markdown(
            f'<div class="result-card">'
            f'<div class="result-label">FRESHNESS STAGE</div>'
            f'<div class="result-value {status_class}">{freshness}</div>'
            f'</div>',
            unsafe_allow_html=True
        )

    with c:
        st.markdown(
            f'<div class="result-card">'
            f'<div class="result-label">MODEL CONFIDENCE</div>'
            f'<div class="result-value">{result["confidence"]*100:.2f}%</div>'
            f'</div>',
            unsafe_allow_html=True
        )

    st.caption(
        "Confidence is the model's softmax score for the selected class. "
        "It is not a guarantee of prediction correctness."
    )

    st.markdown(
        '<div class="section-title">04 · Top 5 predictions</div>',
        unsafe_allow_html=True
    )

    for index, probability in zip(
        result["indices"],
        result["values"]
    ):

        label = clean_label(class_names[index])
        percent = probability * 100

        st.markdown(
            f'<div class="prediction-row">'
            f'<b>{label}</b>'
            f'<span style="float:right;color:#477A52;font-weight:700">'
            f'{percent:.2f}%</span>'
            f'</div>',
            unsafe_allow_html=True
        )

        st.progress(float(probability))

    st.info(
        "The top five probabilities may not add up to 100%, "
        "because the remaining 19 classes are not displayed."
    )

    st.warning(
        "This is a preliminary image-based estimate. "
        "Do not rely on this prediction alone to decide whether "
        "food is safe to eat."
    )


# =====================================================
# FOOTER
# =====================================================

st.markdown("---")

st.markdown(
    '<div class="footer">'
    '🍃 NourishIQ · AI-Based Food Quality and Freshness Intelligence System'
    '<br>Powered by ConvNeXtV2-Base · 24 Classification Classes'
    '</div>',
    unsafe_allow_html=True
)
'''

app_path = "/kaggle/working/app.py"

with open(app_path, "w", encoding="utf-8") as f:
    f.write(app_code)

print("Updated app.py successfully!")
print("File size:", os.path.getsize(app_path), "bytes")

MODEL_PATH = (
    "/kaggle/working/"
    "NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/"
    "results/convnextv2_base/best_convnextv2.pth"
)

print("Checkpoint exists:", os.path.exists(MODEL_PATH))

if os.path.exists(MODEL_PATH):
    print("Checkpoint size:", round(os.path.getsize(MODEL_PATH) / (1024 * 1024), 2), "MB")
else:
    print("WARNING: Model checkpoint not found!")


Updated app.py successfully!
File size: 13176 bytes
Checkpoint exists: True
Checkpoint size: 334.76 MB


In [27]:
import subprocess
import time

subprocess.run(
    ["bash", "-lc", "pkill -f '[s]treamlit run /kaggle/working/app.py' || true"],
    capture_output=True
)

log = open("/kaggle/working/streamlit.log", "w")

process = subprocess.Popen(
    [
        "python", "-m", "streamlit", "run",
        "/kaggle/working/app.py",
        "--server.address=0.0.0.0",
        "--server.port=8501",
        "--server.headless=true"
    ],
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(20)

print("Process running:", process.poll() is None)
print(open("/kaggle/working/streamlit.log").read()[-3000:])

Process running: True


2026-11-22 17:03:14.020 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://136.113.75.135:8501




In [28]:
import requests
print(requests.get("http://127.0.0.1:8501/_stcore/health").text)

ok


In [32]:
!ps aux | grep -E 'streamlit|lt --port' | grep -v grep

In [33]:
!ls -lh /kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/results/convnextv2_base/

total 335M
-rw-r--r-- 1 root root 335M Nov 22 16:35 best_convnextv2.pth
-rw-r--r-- 1 root root 1.7K Nov 22 16:55 classification_report.csv
-rw-r--r-- 1 root root  708 Nov 22 16:55 class_labels.json
-rw-r--r-- 1 root root 2.3K Nov 22 16:55 confusion_matrix.csv


In [34]:
!ps aux | grep streamlit | grep -v grep

In [35]:
!ps aux | grep lt | grep -v grep

root           1  0.0  0.0   2816  1692 ?        Ss   15:26   0:00 /bin/sh -c ENTRYPOINT=jupyter; A0="serverextension"; A1="disable"; A2="caip_notebooks_serverextension"; A3="jupyterlab"; A4="jupyterlab_git"; A5="jupytext"; "$ENTRYPOINT" "$A0" "$A1" "$A2" "$A3" "$A4" "$A5"; mkdir -p "/kaggle/working"; $(expr "$(cat /etc/build_date)" ">=" "20251101" > /dev/null && command -v jupyter-server || command -v jupyter-notebook) --ip='*' --NotebookApp.base_url='/k/372657107/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2IiwidHlwIjoiSldUIn0..Cs9rcO8O3w8Qjib7O4jq8Q.O-W7i4kePybMPBfDcff08xZnBgSAqdPn6ZsxtBVEm8hioKz7NohLPLdGOJS9vqrrf2UFz8fAJ-509DI8dUbT_wfrRzUrFWrWs60p8BFnwpGjJmUrEChV-S8MQp6M6txUyDp4SXsbSTVJRffUPVOutgxZh8fi7kEWhsJNjmZ7wr67u481UrELpuXxTgSgyqlNaueaXH2nxJ3tTb29XGLWCNyk7LYGORTk1VhC2pSOFtN1UPPVpMUdV5W_Nd5lOlOi.MPQ1ka1DrnaPe0DvPtvxIw/proxy/' --NotebookApp.token='' --NotebookApp.notebook_dir="/kaggle/working" --NotebookApp.allow_origin='*' --NotebookApp.disable_check_xsrf=True --NotebookApp.iop

In [36]:
import subprocess
import time
import os

# Start Streamlit server
log_file = open("/kaggle/working/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run",
        "/kaggle/working/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(15)

print("Streamlit process running:", process.poll() is None)

with open("/kaggle/working/streamlit.log") as f:
    print(f.read()[-3000:])

Streamlit process running: True


2026-11-22 17:27:07.227 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://136.113.75.135:8501




In [37]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 1s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

In [39]:
!lsof -i :8501

COMMAND PID USER   FD   TYPE DEVICE SIZE/OFF NODE NAME
python3  47 root   90u  IPv4 228578      0t0  TCP localhost:47348->localhost:8501 (CLOSE_WAIT)


In [51]:
import subprocess
import time
import requests

log_file = open("/kaggle/working/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run",
        "/kaggle/working/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("Streamlit PID:", process.pid)

time.sleep(15)

try:
    response = requests.get("http://127.0.0.1:8501", timeout=10)
    print("Server Status:", response.status_code)
except Exception as e:
    print("Server Error:", e)

Streamlit PID: 41224
Server Status: 200


In [52]:
!lt --port 8501

^C


In [53]:
import requests

try:
    r = requests.get("http://127.0.0.1:8501", timeout=10)
    print("Streamlit:", r.status_code)
except Exception as e:
    print("Error:", e)

Streamlit: 200


In [48]:
!curl -I --max-time 10 http://127.0.0.1:8501

curl: (7) Failed to connect to 127.0.0.1 port 8501 after 0 ms: Couldn't connect to server


In [46]:
!lt --port 8501

^C


In [44]:
!curl -I --max-time 10 http://127.0.0.1:8501

curl: (7) Failed to connect to 127.0.0.1 port 8501 after 0 ms: Couldn't connect to server


In [20]:
app_path = "/kaggle/working/app.py"

old_path = 'MODEL_PATH = "/kaggle/working/best_convnextv2.pth"'

new_path = 'MODEL_PATH = "/kaggle/working/NourishIQ-AI-Based-Food-Quality-and-Freshness-Intelligence-System/results/convnextv2_base/best_convnextv2.pth"'

with open(app_path, "r", encoding="utf-8") as f:
    code = f.read()

code = code.replace(old_path, new_path)

with open(app_path, "w", encoding="utf-8") as f:
    f.write(code)

print("Updated:", new_path in code)

Updated: False


In [21]:
import subprocess

subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

CompletedProcess(args=['pkill', '-f', 'streamlit'], returncode=0, stdout=b'', stderr=b'')

In [22]:
import subprocess
import time

with open("/kaggle/working/streamlit.log", "w") as log:
    process = subprocess.Popen(
        [
            "python", "-m", "streamlit", "run",
            "/kaggle/working/app.py",
            "--server.address=0.0.0.0",
            "--server.port=8501",
            "--server.headless=true"
        ],
        stdout=log,
        stderr=subprocess.STDOUT
    )

time.sleep(20)

print("Process status:", process.poll())
print(open("/kaggle/working/streamlit.log").read()[-5000:])

Process status: None


2026-11-22 16:58:45.115 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://136.113.75.135:8501




In [23]:
import requests

try:
    r = requests.get("http://127.0.0.1:8501/_stcore/health", timeout=10)
    print("Status:", r.status_code)
    print("Response:", r.text)
except Exception as e:
    print("Error:", e)

Status: 200
Response: ok


In [ ]:
!lt --port 8501

In [ ]:
import requests
print(requests.get("https://loca.lt/mytunnelpassword").text)

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file == "best_convnextv2.pth":
            print(os.path.join(root, file))

In [ ]:
#requirements.txt

streamlit
torch
torchvision
timm
numpy
pandas
Pillow


In [ ]:
!streamlit run /kaggle/working/app.py --server.headless true --server.port 8501

In [ ]:
import os

print(os.listdir("/kaggle/working"))